# Train a Custom WordPiece Tokenizer for QQP

This notebook trains an uncased WordPiece tokenizer from scratch on the
**training portion of the Quora Question Pairs dataset**.

The saved tokenizer is designed for three later experiments:

1. LSTM-Attention with a trainable embedding initialized from scratch.
2. ESIM with a trainable embedding initialized from scratch.
3. A custom BERT-style Transformer encoder trained from scratch.

The tokenizer is trained with Hugging Face `tokenizers`, wrapped with
`PreTrainedTokenizerFast`, and saved in a Hugging Face-compatible directory.

The pretrained BERT experiment later must use the tokenizer belonging to
its own pretrained checkpoint, not this custom QQP tokenizer.


## Step 0 — Install dependencies

Run this cell only when these packages are missing. Restart the kernel after
a new installation before continuing.


In [ ]:
# %pip install tokenizers transformers pandas numpy scikit-learn


## Step 1 — Import libraries


In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version as package_version
import json
import os
from pathlib import Path
import platform
import random
from typing import Any

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from tokenizers import Tokenizer
from tokenizers.decoders import WordPiece as WordPieceDecoder
from tokenizers.models import WordPiece
from tokenizers.normalizers import BertNormalizer
from tokenizers.pre_tokenizers import BertPreTokenizer
from tokenizers.processors import TemplateProcessing
from tokenizers.trainers import WordPieceTrainer

from transformers import PreTrainedTokenizerFast


## Step 2 — Define paths

The tokenizer directory contains the normal Hugging Face tokenizer files:

- `tokenizer.json`
- `tokenizer_config.json`
- `vocab.txt`
- optionally `special_tokens_map.json`, depending on the Transformers version

`training_metadata.json` is our own reproducibility file. It combines the
previous `tokenizer_config_custom.json` and `tokenizer_artifacts.json`.


In [ ]:
@dataclass(frozen=True)
class WordPiecePaths:
    """Paths used by the QQP WordPiece-tokenizer pipeline."""

    train_csv_path: Path = Path("data/raw/train.csv")
    processed_dir: Path = Path("data/processed")
    tokenizer_dir: Path = Path(
        "artifacts/tokenizers/wordpiece_uncased_30k"
    )

    @property
    def corpus_path(self) -> Path:
        return self.processed_dir / "tokenizer_corpus.txt"

    @property
    def train_split_path(self) -> Path:
        return self.processed_dir / "train_split.csv"

    @property
    def valid_split_path(self) -> Path:
        return self.processed_dir / "valid_split.csv"

    @property
    def tokenizer_json_path(self) -> Path:
        return self.tokenizer_dir / "tokenizer.json"

    @property
    def tokenizer_config_path(self) -> Path:
        return self.tokenizer_dir / "tokenizer_config.json"

    @property
    def special_tokens_map_path(self) -> Path:
        return self.tokenizer_dir / "special_tokens_map.json"

    @property
    def vocab_path(self) -> Path:
        return self.tokenizer_dir / "vocab.txt"

    @property
    def training_metadata_path(self) -> Path:
        return self.tokenizer_dir / "training_metadata.json"

    def make_dirs(self) -> None:
        """Create all output directories."""
        self.processed_dir.mkdir(parents=True, exist_ok=True)
        self.tokenizer_dir.mkdir(parents=True, exist_ok=True)

    def validate_input(self) -> None:
        """Check that the original QQP training CSV exists."""
        if not self.train_csv_path.is_file():
            raise FileNotFoundError(
                f"QQP training CSV was not found: {self.train_csv_path}"
            )


## Step 3 — Define tokenizer configuration


In [ ]:
@dataclass(frozen=True)
class WordPieceConfig:
    """Configuration for the custom QQP WordPiece tokenizer."""

    vocab_size: int = 30_000
    min_frequency: int = 2

    lowercase: bool = True
    strip_accents: bool = True
    clean_text: bool = True
    handle_chinese_chars: bool = True
    max_input_chars_per_word: int = 100

    # LSTM-Attention and ESIM encode each question separately.
    max_sequence_length: int = 64

    # The custom Transformer encodes the question pair together.
    max_pair_length: int = 128

    pad_token: str = "[PAD]"
    unk_token: str = "[UNK]"
    cls_token: str = "[CLS]"
    sep_token: str = "[SEP]"
    mask_token: str = "[MASK]"

    valid_size: float = 0.10
    seed: int = 28

    continuing_subword_prefix: str = "##"
    length_analysis_batch_size: int = 4_096

    @property
    def special_tokens(self) -> list[str]:
        """Return special tokens in deterministic ID order."""
        return [
            self.pad_token,
            self.unk_token,
            self.cls_token,
            self.sep_token,
            self.mask_token,
        ]


## Step 4 — General utility functions


In [ ]:
def seed_everything(seed: int) -> None:
    """Set deterministic Python and NumPy seeds."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)

    print(f">>> Seed set to {seed}.")


def save_json(data: dict[str, Any], path: str | Path) -> None:
    """Save a dictionary as UTF-8 JSON."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as file:
        json.dump(data, file, indent=4, ensure_ascii=False)

    print(f">>> JSON saved to: {path}")


def load_json(path: str | Path) -> dict[str, Any]:
    """Load a UTF-8 JSON file."""
    path = Path(path)

    with path.open("r", encoding="utf-8") as file:
        data = json.load(file)

    print(f">>> JSON loaded from: {path}")
    return data


def get_package_version(package_name: str) -> str:
    """Return an installed package version for metadata."""
    try:
        return package_version(package_name)
    except PackageNotFoundError:
        return "unknown"


## Step 5 — Load and clean QQP

Corrections compared with the original notebook:

- The label is `is_duplicate`, not `is_duplicated`.
- Required-column checking uses membership, not identity comparison.
- Both questions must be present and non-empty.
- The label must be either `0` or `1`.
- Repeated whitespace is collapsed, but no old word-level preprocessing is used.


In [ ]:
def load_qqp_data(csv_path: str | Path) -> pd.DataFrame:
    """Load QQP and remove rows with invalid questions or labels."""
    csv_path = Path(csv_path)
    df = pd.read_csv(csv_path)

    required_columns = [
        "question1",
        "question2",
        "is_duplicate",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing required QQP columns: {missing_columns}"
        )

    original_rows = len(df)
    df = df[required_columns].copy()

    df["is_duplicate"] = pd.to_numeric(
        df["is_duplicate"],
        errors="coerce",
    )

    df = df.dropna(
        subset=[
            "question1",
            "question2",
            "is_duplicate",
        ]
    ).copy()

    for column in ["question1", "question2"]:
        df[column] = (
            df[column]
            .astype(str)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

    valid_questions = (
        df["question1"].ne("")
        & df["question2"].ne("")
    )
    valid_labels = df["is_duplicate"].isin([0, 1])

    df = df.loc[
        valid_questions & valid_labels,
        required_columns,
    ].copy()

    df["is_duplicate"] = (
        df["is_duplicate"].astype(np.int8)
    )
    df = df.reset_index(drop=True)

    removed_rows = original_rows - len(df)

    print(
        f">>> QQP loaded from: {csv_path}\n"
        f">>> Original rows: {original_rows:,}\n"
        f">>> Valid rows: {len(df):,}\n"
        f">>> Removed invalid rows: {removed_rows:,}"
    )

    return df


## Step 6 — Create and save train/validation splits

The tokenizer corpus is built only from `train_df`. Validation questions do
not participate in vocabulary learning.


In [ ]:
def create_train_valid_split(
    df: pd.DataFrame,
    cfg: WordPieceConfig,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Create reproducible stratified QQP splits."""
    train_df, valid_df = train_test_split(
        df,
        test_size=cfg.valid_size,
        random_state=cfg.seed,
        shuffle=True,
        stratify=df["is_duplicate"],
    )

    train_df = train_df.reset_index(drop=True)
    valid_df = valid_df.reset_index(drop=True)

    print(
        ">>> Stratified split created.\n"
        f">>> Training rows: {len(train_df):,}\n"
        f">>> Validation rows: {len(valid_df):,}\n"
        f">>> Train duplicate rate: "
        f"{train_df['is_duplicate'].mean():.4f}\n"
        f">>> Validation duplicate rate: "
        f"{valid_df['is_duplicate'].mean():.4f}"
    )

    return train_df, valid_df


def save_splits(
    train_df: pd.DataFrame,
    valid_df: pd.DataFrame,
    paths: WordPiecePaths,
) -> None:
    """Save the exact splits reused by later models."""
    train_df.to_csv(
        paths.train_split_path,
        index=False,
    )
    valid_df.to_csv(
        paths.valid_split_path,
        index=False,
    )

    print(
        f">>> Training split saved to: "
        f"{paths.train_split_path}\n"
        f">>> Validation split saved to: "
        f"{paths.valid_split_path}"
    )


## Step 7 — Build the tokenizer-training corpus

Each training question is written on its own line. Duplicate questions are
intentionally preserved because their frequency is meaningful to vocabulary
training.


In [ ]:
def build_corpus_file(
    train_df: pd.DataFrame,
    paths: WordPiecePaths,
) -> int:
    """Write Q1 and Q2 from the training split to a text corpus."""
    question_count = 0

    with paths.corpus_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        for row in train_df.itertuples(index=False):
            file.write(f"{row.question1}\n")
            file.write(f"{row.question2}\n")
            question_count += 2

    print(
        f">>> Tokenizer corpus saved to: "
        f"{paths.corpus_path}\n"
        f">>> Questions written: {question_count:,}"
    )

    return question_count


## Step 8 — Train the low-level WordPiece backend

The pipeline is:

`BertNormalizer → BertPreTokenizer → WordPiece → TemplateProcessing → WordPieceDecoder`

Pair inputs are represented as:

`[CLS] question1 [SEP] question2 [SEP]`

Question 1 uses token-type ID `0`; question 2 and its final `[SEP]` use ID `1`.


In [ ]:
def train_wordpiece_backend(
    paths: WordPiecePaths,
    cfg: WordPieceConfig,
) -> Tokenizer:
    """Train the Rust-backed WordPiece tokenizer."""
    tokenizer = Tokenizer(
        WordPiece(
            unk_token=cfg.unk_token,
            max_input_chars_per_word=(
                cfg.max_input_chars_per_word
            ),
            continuing_subword_prefix=(
                cfg.continuing_subword_prefix
            ),
        )
    )

    tokenizer.normalizer = BertNormalizer(
        clean_text=cfg.clean_text,
        handle_chinese_chars=(
            cfg.handle_chinese_chars
        ),
        strip_accents=cfg.strip_accents,
        lowercase=cfg.lowercase,
    )

    tokenizer.pre_tokenizer = BertPreTokenizer()

    trainer = WordPieceTrainer(
        vocab_size=cfg.vocab_size,
        min_frequency=cfg.min_frequency,
        show_progress=True,
        special_tokens=cfg.special_tokens,
        continuing_subword_prefix=(
            cfg.continuing_subword_prefix
        ),
    )

    tokenizer.train(
        files=[str(paths.corpus_path)],
        trainer=trainer,
    )

    special_token_ids = {
        token: tokenizer.token_to_id(token)
        for token in cfg.special_tokens
    }

    missing_special_tokens = [
        token
        for token, token_id
        in special_token_ids.items()
        if token_id is None
    ]

    if missing_special_tokens:
        raise RuntimeError(
            "The trained vocabulary is missing "
            f"special tokens: {missing_special_tokens}"
        )

    tokenizer.post_processor = TemplateProcessing(
        single=(
            f"{cfg.cls_token} $A "
            f"{cfg.sep_token}"
        ),
        pair=(
            f"{cfg.cls_token} $A "
            f"{cfg.sep_token} "
            f"$B:1 {cfg.sep_token}:1"
        ),
        special_tokens=[
            (
                cfg.cls_token,
                special_token_ids[cfg.cls_token],
            ),
            (
                cfg.sep_token,
                special_token_ids[cfg.sep_token],
            ),
        ],
    )

    tokenizer.decoder = WordPieceDecoder(
        prefix=cfg.continuing_subword_prefix,
        cleanup=True,
    )

    print(
        ">>> WordPiece backend trained.\n"
        f">>> Actual vocabulary size: "
        f"{tokenizer.get_vocab_size():,}"
    )

    return tokenizer


## Step 9 — Wrap it with `PreTrainedTokenizerFast`

The tokenizer is still trained from scratch. `PreTrainedTokenizerFast` is
only the generic Transformers-compatible interface around the Rust backend.


In [ ]:
def create_fast_tokenizer(
    backend_tokenizer: Tokenizer,
    cfg: WordPieceConfig,
) -> PreTrainedTokenizerFast:
    """Create the Transformers-compatible tokenizer wrapper."""
    tokenizer = PreTrainedTokenizerFast(
        tokenizer_object=backend_tokenizer,
        unk_token=cfg.unk_token,
        pad_token=cfg.pad_token,
        cls_token=cfg.cls_token,
        sep_token=cfg.sep_token,
        mask_token=cfg.mask_token,
        model_max_length=cfg.max_pair_length,
    )

    tokenizer.padding_side = "right"
    tokenizer.truncation_side = "right"

    if not tokenizer.is_fast:
        raise RuntimeError(
            "Expected a fast tokenizer backend."
        )

    print(
        ">>> Transformers wrapper created.\n"
        f">>> Wrapper class: "
        f"{type(tokenizer).__name__}\n"
        f">>> Vocabulary size: {len(tokenizer):,}"
    )

    return tokenizer


## Step 10 — Save Hugging Face-compatible files

`save_pretrained()` saves the complete Transformers tokenizer state.
The low-level WordPiece model is also exported explicitly so that
`vocab.txt` is always present.


In [ ]:
def save_tokenizer(
    backend_tokenizer: Tokenizer,
    tokenizer: PreTrainedTokenizerFast,
    paths: WordPiecePaths,
) -> list[str]:
    """Save the complete tokenizer and WordPiece vocabulary."""
    paths.tokenizer_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    # Guarantees a complete unified tokenizer.json.
    backend_tokenizer.save(
        str(paths.tokenizer_json_path)
    )

    # Writes tokenizer.json, tokenizer_config.json,
    # and special-token information.
    tokenizer.save_pretrained(
        str(paths.tokenizer_dir)
    )

    # Guarantees the standard WordPiece vocab.txt.
    backend_tokenizer.model.save(
        str(paths.tokenizer_dir)
    )

    required_files = [
        paths.tokenizer_json_path,
        paths.tokenizer_config_path,
        paths.vocab_path,
    ]

    missing_files = [
        str(path)
        for path in required_files
        if not path.is_file()
    ]

    if missing_files:
        raise FileNotFoundError(
            "Tokenizer saving completed, but "
            f"required files are missing: {missing_files}"
        )

    saved_files = sorted(
        path.name
        for path in paths.tokenizer_dir.iterdir()
        if path.is_file()
    )

    print(">>> Tokenizer files saved:")
    for filename in saved_files:
        print(f"    - {filename}")

    if not paths.special_tokens_map_path.exists():
        print(
            ">>> Note: special_tokens_map.json was not "
            "emitted by this Transformers version. "
            "The special-token definitions are still "
            "stored in the saved tokenizer state."
        )

    return saved_files


## Step 11 — Validate separate and paired encodings

- LSTM-Attention and ESIM: separate questions, no `[CLS]` or `[SEP]`.
- Custom Transformer: one paired sequence with `[CLS]` and two `[SEP]` tokens.


In [ ]:
def validate_tokenizer(
    tokenizer: PreTrainedTokenizerFast,
    cfg: WordPieceConfig,
) -> None:
    """Validate special tokens and both model input formats."""
    special_token_ids = {
        "pad": tokenizer.pad_token_id,
        "unk": tokenizer.unk_token_id,
        "cls": tokenizer.cls_token_id,
        "sep": tokenizer.sep_token_id,
        "mask": tokenizer.mask_token_id,
    }

    if any(
        token_id is None
        for token_id in special_token_ids.values()
    ):
        raise ValueError(
            "One or more special-token IDs are missing: "
            f"{special_token_ids}"
        )

    if len(set(special_token_ids.values())) != len(
        special_token_ids
    ):
        raise ValueError(
            "Special-token IDs are not unique: "
            f"{special_token_ids}"
        )

    question1 = "How can I learn machine learning?"
    question2 = (
        "What is the best way to study "
        "machine learning?"
    )

    separate_encoding = tokenizer(
        question1,
        add_special_tokens=False,
        truncation=True,
        max_length=cfg.max_sequence_length,
        padding="max_length",
        return_attention_mask=True,
        return_token_type_ids=False,
    )

    pair_encoding = tokenizer(
        question1,
        question2,
        add_special_tokens=True,
        truncation=True,
        max_length=cfg.max_pair_length,
        padding="max_length",
        return_attention_mask=True,
        return_token_type_ids=True,
    )

    if (
        len(separate_encoding["input_ids"])
        != cfg.max_sequence_length
    ):
        raise AssertionError(
            "Separate-question length is incorrect."
        )

    if (
        len(pair_encoding["input_ids"])
        != cfg.max_pair_length
    ):
        raise AssertionError(
            "Question-pair length is incorrect."
        )

    active_length = int(
        sum(pair_encoding["attention_mask"])
    )
    active_ids = pair_encoding[
        "input_ids"
    ][:active_length]
    active_type_ids = pair_encoding[
        "token_type_ids"
    ][:active_length]

    active_tokens = (
        tokenizer.convert_ids_to_tokens(active_ids)
    )

    if active_tokens[0] != cfg.cls_token:
        raise AssertionError(
            "Pair encoding must begin with [CLS]."
        )

    if active_tokens.count(cfg.sep_token) != 2:
        raise AssertionError(
            "Pair encoding must contain two [SEP] tokens."
        )

    first_sep_index = active_tokens.index(
        cfg.sep_token
    )

    if any(
        type_id != 0
        for type_id in active_type_ids[
            : first_sep_index + 1
        ]
    ):
        raise AssertionError(
            "Question 1 and its special tokens "
            "must use token-type ID 0."
        )

    if any(
        type_id != 1
        for type_id in active_type_ids[
            first_sep_index + 1 :
        ]
    ):
        raise AssertionError(
            "Question 2 and its final [SEP] "
            "must use token-type ID 1."
        )

    print(">>> Special-token IDs:")
    print(special_token_ids)

    print(
        "\n>>> Separate-question WordPiece tokens:"
    )
    print(tokenizer.tokenize(question1))

    print("\n>>> Active paired tokens:")
    print(active_tokens)

    print("\n>>> Active token-type IDs:")
    print(active_type_ids)

    print("\n>>> Tokenizer validation passed.")


## Step 12 — Verify save/reload consistency


In [ ]:
def verify_reload_consistency(
    tokenizer: PreTrainedTokenizerFast,
    paths: WordPiecePaths,
    cfg: WordPieceConfig,
) -> PreTrainedTokenizerFast:
    """Reload the tokenizer and verify identical encodings."""
    reloaded_tokenizer = (
        PreTrainedTokenizerFast.from_pretrained(
            str(paths.tokenizer_dir),
            local_files_only=True,
        )
    )

    question1 = "Why is the sky blue?"
    question2 = (
        "What causes the sky to appear blue?"
    )

    encoding_kwargs = {
        "add_special_tokens": True,
        "truncation": True,
        "max_length": cfg.max_pair_length,
        "padding": "max_length",
        "return_attention_mask": True,
        "return_token_type_ids": True,
    }

    original = tokenizer(
        question1,
        question2,
        **encoding_kwargs,
    )
    reloaded = reloaded_tokenizer(
        question1,
        question2,
        **encoding_kwargs,
    )

    for field in [
        "input_ids",
        "attention_mask",
        "token_type_ids",
    ]:
        if original[field] != reloaded[field]:
            raise AssertionError(
                f"Reloaded tokenizer changed {field}."
            )

    if (
        tokenizer.special_tokens_map
        != reloaded_tokenizer.special_tokens_map
    ):
        raise AssertionError(
            "Reloading changed the special-token map."
        )

    print(
        ">>> Save/reload consistency test passed."
    )

    return reloaded_tokenizer


## Step 13 — Analyze sequence lengths

The proposed limits of `64` and `128` are checked empirically after tokenizer
training. This analysis uses the training split only and processes the data
in batches.


In [ ]:
def summarize_lengths(
    lengths: np.ndarray,
    limit: int,
) -> dict[str, int | float]:
    """Summarize one token-length distribution."""
    percentiles = np.percentile(
        lengths,
        [50, 90, 95, 99],
    )

    over_limit = int(
        np.count_nonzero(lengths > limit)
    )

    return {
        "count": int(lengths.size),
        "mean": round(float(lengths.mean()), 3),
        "p50": int(np.ceil(percentiles[0])),
        "p90": int(np.ceil(percentiles[1])),
        "p95": int(np.ceil(percentiles[2])),
        "p99": int(np.ceil(percentiles[3])),
        "max": int(lengths.max()),
        "configured_limit": int(limit),
        "count_over_limit": over_limit,
        "percent_over_limit": round(
            100.0 * over_limit / lengths.size,
            4,
        ),
    }


def analyze_sequence_lengths(
    backend_tokenizer: Tokenizer,
    train_df: pd.DataFrame,
    cfg: WordPieceConfig,
) -> tuple[dict[str, Any], pd.DataFrame]:
    """Measure separate-question and pair token lengths."""
    q1_lengths: list[int] = []
    q2_lengths: list[int] = []
    pair_lengths: list[int] = []

    batch_size = cfg.length_analysis_batch_size

    for start in range(
        0,
        len(train_df),
        batch_size,
    ):
        batch = train_df.iloc[
            start : start + batch_size
        ]

        question1 = batch["question1"].tolist()
        question2 = batch["question2"].tolist()

        q1_encodings = (
            backend_tokenizer.encode_batch(
                question1,
                add_special_tokens=False,
            )
        )
        q2_encodings = (
            backend_tokenizer.encode_batch(
                question2,
                add_special_tokens=False,
            )
        )
        pair_encodings = (
            backend_tokenizer.encode_batch(
                list(zip(question1, question2)),
                add_special_tokens=True,
            )
        )

        q1_lengths.extend(
            len(encoding.ids)
            for encoding in q1_encodings
        )
        q2_lengths.extend(
            len(encoding.ids)
            for encoding in q2_encodings
        )
        pair_lengths.extend(
            len(encoding.ids)
            for encoding in pair_encodings
        )

    q1_array = np.asarray(
        q1_lengths,
        dtype=np.int32,
    )
    q2_array = np.asarray(
        q2_lengths,
        dtype=np.int32,
    )
    pair_array = np.asarray(
        pair_lengths,
        dtype=np.int32,
    )

    all_questions = np.concatenate(
        [q1_array, q2_array]
    )

    rows_over_separate_limit = int(
        np.count_nonzero(
            (
                q1_array
                > cfg.max_sequence_length
            )
            | (
                q2_array
                > cfg.max_sequence_length
            )
        )
    )

    statistics = {
        "question1": summarize_lengths(
            q1_array,
            cfg.max_sequence_length,
        ),
        "question2": summarize_lengths(
            q2_array,
            cfg.max_sequence_length,
        ),
        "all_separate_questions": (
            summarize_lengths(
                all_questions,
                cfg.max_sequence_length,
            )
        ),
        "paired_inputs": summarize_lengths(
            pair_array,
            cfg.max_pair_length,
        ),
        "rows_with_either_question_over_"
        "separate_limit": {
            "count": rows_over_separate_limit,
            "percent": round(
                100.0
                * rows_over_separate_limit
                / len(train_df),
                4,
            ),
        },
    }

    table = pd.DataFrame(
        {
            "separate_questions": (
                statistics[
                    "all_separate_questions"
                ]
            ),
            "paired_inputs": (
                statistics["paired_inputs"]
            ),
        }
    ).T

    print(
        ">>> Sequence-length analysis completed."
    )
    print(table.to_string())

    return statistics, table


## Step 14 — Build the single custom metadata file

This file records both the training recipe and the resulting tokenizer
statistics. Hugging Face does not require it, but it makes the experiment
reproducible.


In [ ]:
def build_training_metadata(
    cfg: WordPieceConfig,
    paths: WordPiecePaths,
    train_df: pd.DataFrame,
    valid_df: pd.DataFrame,
    corpus_question_count: int,
    tokenizer: PreTrainedTokenizerFast,
    length_statistics: dict[str, Any],
    saved_files: list[str],
) -> dict[str, Any]:
    """Build reproducibility metadata for the tokenizer run."""
    all_saved_files = sorted(
        set(
            saved_files
            + [paths.training_metadata_path.name]
        )
    )

    return {
        "created_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "implementation": {
            "algorithm": "WordPiece",
            "backend": (
                "Hugging Face Tokenizers"
            ),
            "wrapper": type(tokenizer).__name__,
        },
        "library_versions": {
            "python": platform.python_version(),
            "numpy": get_package_version("numpy"),
            "pandas": get_package_version("pandas"),
            "scikit_learn": get_package_version(
                "scikit-learn"
            ),
            "tokenizers": get_package_version(
                "tokenizers"
            ),
            "transformers": get_package_version(
                "transformers"
            ),
        },
        "paths": {
            "source_csv": str(
                paths.train_csv_path
            ),
            "training_split": str(
                paths.train_split_path
            ),
            "validation_split": str(
                paths.valid_split_path
            ),
            "training_corpus": str(
                paths.corpus_path
            ),
            "tokenizer_directory": str(
                paths.tokenizer_dir
            ),
        },
        "configuration": {
            **asdict(cfg),
            "special_tokens": cfg.special_tokens,
        },
        "data": {
            "train_rows": int(len(train_df)),
            "validation_rows": int(
                len(valid_df)
            ),
            "corpus_questions": int(
                corpus_question_count
            ),
            "tokenizer_training_source": (
                "question1 and question2 from "
                "the training split only"
            ),
        },
        "training_results": {
            "actual_vocab_size": int(
                len(tokenizer)
            ),
            "special_token_ids": {
                "pad_token_id": (
                    tokenizer.pad_token_id
                ),
                "unk_token_id": (
                    tokenizer.unk_token_id
                ),
                "cls_token_id": (
                    tokenizer.cls_token_id
                ),
                "sep_token_id": (
                    tokenizer.sep_token_id
                ),
                "mask_token_id": (
                    tokenizer.mask_token_id
                ),
            },
            "saved_files": all_saved_files,
        },
        "sequence_length_analysis": (
            length_statistics
        ),
    }


## Step 15 — Run the complete pipeline

This cell performs every stage in the correct order.


In [ ]:
cfg = WordPieceConfig()
paths = WordPiecePaths()

paths.make_dirs()
paths.validate_input()
seed_everything(cfg.seed)

df = load_qqp_data(
    paths.train_csv_path
)

train_df, valid_df = create_train_valid_split(
    df=df,
    cfg=cfg,
)

save_splits(
    train_df=train_df,
    valid_df=valid_df,
    paths=paths,
)

corpus_question_count = build_corpus_file(
    train_df=train_df,
    paths=paths,
)

backend_tokenizer = train_wordpiece_backend(
    paths=paths,
    cfg=cfg,
)

tokenizer = create_fast_tokenizer(
    backend_tokenizer=backend_tokenizer,
    cfg=cfg,
)

saved_files = save_tokenizer(
    backend_tokenizer=backend_tokenizer,
    tokenizer=tokenizer,
    paths=paths,
)

validate_tokenizer(
    tokenizer=tokenizer,
    cfg=cfg,
)

tokenizer = verify_reload_consistency(
    tokenizer=tokenizer,
    paths=paths,
    cfg=cfg,
)

length_statistics, length_table = (
    analyze_sequence_lengths(
        backend_tokenizer=backend_tokenizer,
        train_df=train_df,
        cfg=cfg,
    )
)

training_metadata = build_training_metadata(
    cfg=cfg,
    paths=paths,
    train_df=train_df,
    valid_df=valid_df,
    corpus_question_count=(
        corpus_question_count
    ),
    tokenizer=tokenizer,
    length_statistics=length_statistics,
    saved_files=saved_files,
)

save_json(
    training_metadata,
    paths.training_metadata_path,
)

print(
    "\n>>> WordPiece tokenizer pipeline completed."
)


## Step 16 — Final compatibility check


In [ ]:
required_files = [
    paths.tokenizer_json_path,
    paths.tokenizer_config_path,
    paths.vocab_path,
    paths.training_metadata_path,
]

missing_files = [
    str(path)
    for path in required_files
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        f"Missing final artifacts: {missing_files}"
    )

final_tokenizer = (
    PreTrainedTokenizerFast.from_pretrained(
        str(paths.tokenizer_dir),
        local_files_only=True,
    )
)

sample = final_tokenizer(
    "Why is the sky blue?",
    "What makes the sky appear blue?",
    return_token_type_ids=True,
)

print(sample)
print(
    "\n>>> Final tokenizer is locally reloadable "
    "with from_pretrained()."
)


## Step 17 — How later models will use this tokenizer

### LSTM-Attention and ESIM

Encode the two questions separately with no `[CLS]` or `[SEP]`:

```python
q1 = tokenizer(
    question1_batch,
    add_special_tokens=False,
    padding="max_length",
    truncation=True,
    max_length=cfg.max_sequence_length,
    return_tensors="pt",
)

q2 = tokenizer(
    question2_batch,
    add_special_tokens=False,
    padding="max_length",
    truncation=True,
    max_length=cfg.max_sequence_length,
    return_tensors="pt",
)
```

Their embedding layer is initialized from scratch:

```python
embedding = nn.Embedding(
    num_embeddings=len(tokenizer),
    embedding_dim=embedding_dim,
    padding_idx=tokenizer.pad_token_id,
)
```

### Custom BERT-style Transformer

Encode the pair together:

```python
pair = tokenizer(
    question1_batch,
    question2_batch,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=cfg.max_pair_length,
    return_token_type_ids=True,
    return_tensors="pt",
)
```

The pretrained BERT experiment later uses its own pretrained tokenizer.


## Expected final directory

```text
artifacts/tokenizers/wordpiece_uncased_30k/
├── tokenizer.json
├── tokenizer_config.json
├── vocab.txt
├── special_tokens_map.json   # version-dependent
└── training_metadata.json
```
